In [ ]:
# ============================================================================
# ANOMALY DETECTION PROJECT - MACHINE LEARNING PIPELINE
# ============================================================================
# This notebook implements a comprehensive machine learning pipeline for
# anomaly detection using multiple classification algorithms

# ============================================================================
# STEP 1: IMPORT LIBRARIES AND LOAD DATA
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, BaggingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, roc_curve

# Import necessary modules for specific models
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
import psutil  # For monitoring system performance
import tracemalloc  # For memory profiling
import time

print("="*60)
print("LOADING DATA AND INITIAL EXPLORATION")
print("="*60)

# Load the training data
# IMPORTANT: Update csv_path with your actual data file path
csv_path = r''  # Replace with path to train_features.csv
df = pd.read_csv(csv_path)

# ============================================================================
# STEP 2: DATA PREPROCESSING AND EXPLORATION
# ============================================================================

# Separate features (X) and target variable (y)
# Remove non-numeric/useless columns except the target
X = df.drop(columns=[])  # Remove unnecessary columns if needed
y = df['anomaly']  # Target variable: 1 = Anomaly, 0 = Normal

# Display data information
print(f"\n✓ Data loaded successfully")
print(f"Shape of X (features): {X.shape}")
print(f"Shape of y (target): {y.shape}")
print(f"Columns in X: {X.columns.tolist()[:10]}... (showing first 10)")

# Identify numeric and categorical features
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"\nFeature types:")
print(f"  - Numeric features: {len(numeric_features)}")
print(f"  - Categorical features: {len(categorical_features)}")

# ============================================================================
# STEP 3: HANDLE MISSING VALUES
# ============================================================================

print("\n" + "="*60)
print("NULL VALUES IMPUTATION")
print("="*60)

# Check null values before imputation
nulls_before = X.isnull().sum().sum()
total_values = X.shape[0] * X.shape[1]
nulls_percentage_before = 100 * nulls_before / total_values
print(f"Null values before imputation: {nulls_before} ({nulls_percentage_before:.2f}% of total)")

# Impute missing numeric values using median strategy
# Median is robust to outliers
numeric_imputer = SimpleImputer(strategy='median')
X[numeric_features] = numeric_imputer.fit_transform(X[numeric_features])

# Verify imputation
nulls_after = X.isnull().sum().sum()
print(f"Null values after imputation: {nulls_after}")

# ============================================================================
# STEP 4: SPLIT DATA INTO TRAIN AND TEST SETS
# ============================================================================

# Stratified split: maintains class distribution
# Train: 70%, Test: 30%
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.3, 
    random_state=42, 
    stratify=y
)

print(f"\n✓ Data split completed successfully")
print(f"Training set size: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set size: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"Features: {X_train.shape[1]}")


In [ ]:
# Display training data preview
print("Training Data Preview (first 5 rows):")
X_train.head()


In [ ]:
# ============================================================================
# STEP 5: VISUALIZE CLASS DISTRIBUTION
# ============================================================================

import matplotlib.pyplot as plt

print("="*60)
print("ANALYZING CLASS DISTRIBUTION")
print("="*60)

# Count instances in each class
fraud_counts = y.value_counts().sort_index()
labels = ['Normal', 'Anomaly']
colors = ['#2ecc71', '#e74c3c']
total = fraud_counts.sum()
percentages = [100 * v / total for v in fraud_counts.values]

# Create bar chart
fig, ax = plt.subplots(figsize=(5, 8))
bars = ax.bar(labels, fraud_counts.values, color=colors)

# Customize chart
ax.set_title('Distribution of Normal vs Anomaly in DCuba', fontsize=20, fontweight='bold')
ax.set_ylabel('Number of Records', fontweight='bold', fontsize=18)
ax.set_xlabel('Target Feature: anomaly', fontweight='bold', fontsize=18)
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.tick_params(axis='both', which='major', labelsize=20)

# Add value labels on bars
for i, (bar, count, pct) in enumerate(zip(bars, fraud_counts.values, percentages)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(count)}\n({pct:.1f}%)',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Print class distribution statistics
print(f"\nClass Distribution Statistics:")
for label, count, pct in zip(labels, fraud_counts.values, percentages):
    print(f"  {label:15s}: {int(count):7d} samples ({pct:5.2f}%)")


In [ ]:
# ============================================================================
# STEP 6: INITIALIZE CLASSIFICATION MODELS
# ============================================================================

print("\n" + "="*60)
print("INITIALIZING MACHINE LEARNING MODELS")
print("="*60)

# Create instances of all classification models with optimized parameters
# Hyperparameters are tuned for anomaly detection tasks

# Ensemble Methods
model_rf = RandomForestClassifier(
    n_estimators=250,
    class_weight='balanced',  # Handle class imbalance
    random_state=42
)

# Neural Network
model_MLP = MLPClassifier(
    hidden_layer_sizes=(100,),
    max_iter=1000,
    random_state=42
)

# Linear Models
model_lr = LogisticRegression(
    class_weight='balanced',  # Handle class imbalance
    max_iter=1000,
    random_state=42
)

# Distance-based Model
model_knn = KNeighborsClassifier()

# Gradient-based Ensemble
model_gb = GradientBoostingClassifier(
    random_state=42
)

# Extreme Gradient Boosting
model_xgb = XGBClassifier(
    n_estimators=250,
    random_state=42
)

# Bagging Ensemble
model_bagging = BaggingClassifier(
    n_estimators=250,
    random_state=42
)

# Linear Discriminant Analysis
model_lda = LinearDiscriminantAnalysis()

# Probabilistic Model
model_nb = GaussianNB()

print("✓ All models initialized successfully")


In [ ]:
# ============================================================================
# STEP 7: CREATE PREPROCESSING PIPELINES
# ============================================================================

print("\n" + "="*60)
print("CONFIGURING PREPROCESSING PIPELINES")
print("="*60)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

print(f"\nFeature Summary:")
print(f"  Numeric columns: {len(numeric_features)}")
print(f"  Categorical columns: {len(categorical_features)}")

if len(categorical_features) > 0:
    print(f"  Detected categorical features: {categorical_features}")
    
    # PIPELINE WITH SCALING
    # For models sensitive to feature magnitude (KNN, LR, SVM, LDA, MLP)
    preprocessor_scaled = ColumnTransformer(
        transformers=[
            # StandardScaler normalizes numeric features to mean=0, std=1
            ('num', StandardScaler(), numeric_features),
            # OneHotEncoder converts categorical to binary dummy variables
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False, max_categories=10), categorical_features)
        ],
        remainder='drop'  # Drop any other columns
    )
    
    # PIPELINE WITHOUT SCALING
    # For tree-based models (RF, XGB, GBM, Bagging) which are scale-invariant
    preprocessor_no_scaling = ColumnTransformer(
        transformers=[
            # Keep numeric features as-is
            ('num', 'passthrough', numeric_features),
            # OneHotEncode categorical features
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False, max_categories=10), categorical_features)
        ],
        remainder='drop'
    )
else:
    # Only numeric features case
    print("  No categorical features detected - using only numeric preprocessing")
    
    preprocessor_scaled = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features)
        ],
        remainder='drop'
    )
    
    preprocessor_no_scaling = ColumnTransformer(
        transformers=[
            ('num', 'passthrough', numeric_features)
        ],
        remainder='drop'
    )

print("✓ Preprocessors configured successfully")
print("  - preprocessor_scaled: For distance-based & linear models")
print("  - preprocessor_no_scaling: For tree-based models")


In [ ]:
# NOTE: Redefine preprocessors here to ensure fresh instances for pipeline creation
# This prevents any shared state issues between different pipeline configurations

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Preprocessor with scaling (StandardScaler)
# Used for: Logistic Regression, KNN, LDA, Naive Bayes, MLP
preprocessor_scaled = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),  # Normalize: (x - mean) / std
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

# Preprocessor WITHOUT scaling (Passthrough)
# Used for: Random Forest, XGBoost, Gradient Boosting, Bagging
preprocessor_no_scaling = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features),  # Keep original values
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

print("✓ Preprocessor instances recreated for pipeline composition")


In [ ]:
# ============================================================================
# STEP 8: CREATE ML PIPELINES WITH PREPROCESSING & CLASS BALANCING
# ============================================================================

print("\n" + "="*60)
print("CREATING MACHINE LEARNING PIPELINES")
print("="*60)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# SMOTE (Synthetic Minority Over-sampling Technique)
# Generates synthetic samples for minority class to handle imbalance
# sampling_strategy=0.4: target 40% minority class after generation
smote = SMOTE(sampling_strategy=0.4, random_state=42)

# Dictionary of all model pipelines
# Each pipeline: Preprocessor -> SMOTE (if needed) -> Classifier
pipelines = {
    # ========== NEURAL NETWORKS ==========
    'MLP': ImbPipeline(steps=[
        ('preprocessor', preprocessor_scaled),
        ('smote', smote),  # Balance classes
        ('classifier', model_MLP)
    ]),

    # ========== LINEAR MODELS ==========
    'Logistic Regression': ImbPipeline(steps=[
        ('preprocessor', preprocessor_scaled),
        ('smote', smote),  # Balance classes
        ('classifier', model_lr)
    ]),

    # ========== ENSEMBLE METHODS - Tree-based ==========
    'Random Forest': ImbPipeline(steps=[
        ('preprocessor', preprocessor_no_scaling),
        # No SMOTE: RF handles imbalance via class_weight parameter
        ('classifier', model_rf)
    ]),

    'KNN': ImbPipeline(steps=[
        ('preprocessor', preprocessor_scaled),
        ('smote', smote),  # Balance classes
        ('classifier', model_knn)
    ]),

    'Gradient Boosting': ImbPipeline(steps=[
        ('preprocessor', preprocessor_no_scaling),
        # No SMOTE: GBM is generally stable with imbalanced data
        ('classifier', model_gb)
    ]),

    'Bagging': ImbPipeline(steps=[
        ('preprocessor', preprocessor_no_scaling),
        # No SMOTE: Bagging has built-in mechanisms
        ('classifier', model_bagging)
    ]),

    # ========== DISCRIMINANT ANALYSIS ==========
    'LDA': ImbPipeline(steps=[
        ('preprocessor', preprocessor_scaled),
        ('classifier', model_lda)
    ]),

    # ========== PROBABILISTIC MODELS ==========
    'Naive Bayes': ImbPipeline(steps=[
        ('preprocessor', preprocessor_scaled),
        ('classifier', model_nb)
    ]),

    # ========== GRADIENT BOOSTING VARIANTS ==========
    'XGBoost': ImbPipeline(steps=[
        ('preprocessor', preprocessor_no_scaling),
        # No SMOTE: XGBoost has built-in class weight handling
        ('classifier', model_xgb)
    ])
}

print(f"✓ {len(pipelines)} pipelines created successfully")
for model_name in pipelines.keys():
    print(f"  - {model_name}")


In [ ]:
# ============================================================================
# STEP 9: CROSS-VALIDATION WITH PERFORMANCE & EFFICIENCY METRICS
# ============================================================================

print("\n" + "="*60)
print("SETTING UP CROSS-VALIDATION")
print("="*60)

from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, matthews_corrcoef, accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, roc_curve, auc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report

# StratifiedKFold: Maintains class distribution in each fold
# 10 folds: common choice balancing bias-variance and computational cost
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Define scoring metrics for evaluation
# Each metric provides different insights into model performance
scorers = {
    'accuracy': make_scorer(accuracy_score),           # Overall correctness
    'f1_weighted': make_scorer(f1_score, average='weighted'),  # Harmonic mean (weighted)
    'precision': make_scorer(precision_score, average='weighted'),  # True positives / predicted positives
    'recall': make_scorer(recall_score, average='weighted'),  # True positives / actual positives
    'roc_auc': make_scorer(roc_auc_score, response_method='predict_proba'),  # Area under ROC curve
    'mcc': make_scorer(matthews_corrcoef)  # Matthews Correlation Coefficient (best for imbalanced)
}

# Dictionary to store efficiency metrics
efficiency_metrics = {
    'training_time': [],      # Wall time in seconds
    'cpu_percent': [],        # CPU utilization percentage
    'ram_mb': [],             # Peak RAM usage in MB
    'process_rss_mb': []      # Process RSS (resident set size)
}

# List to store cross-validation results
cv_results = []

# ============================================================================
# EFFICIENCY MEASUREMENT FUNCTION
# ============================================================================

def measure_efficiency(func, *args, **kwargs):
    """
    Measure time, CPU, and memory usage of a function with high accuracy.
    
    Parameters:
    -----------
    func : callable
        Function to measure
    *args, **kwargs : arguments to pass to func
    
    Returns:
    --------
    dict : Contains execution results and performance metrics
        - 'result': Function output
        - 'time': Wall time in seconds (perf_counter)
        - 'cpu_percent': CPU usage as % of wall time
        - 'cpu_time': Actual CPU time used
        - 'ram_mb': Peak memory from tracemalloc
        - 'process_rss_mb': Process RSS memory
        - 'delta_rss': Change in RSS
        - 'system_ram_percent': System-wide RAM usage %
    """
    p = psutil.Process()
    
    # Capture initial system metrics
    ram_before = psutil.virtual_memory()
    rss_before = p.memory_info().rss / (1024 * 1024)  # Convert bytes to MB
    
    # Use perf_counter for high-resolution wall time measurement
    # More accurate than time.time() for short intervals
    start_time = time.perf_counter()
    cpu_times_start = p.cpu_times()
    
    # Start memory tracking
    tracemalloc.start()
    
    # Execute the function
    result = func(*args, **kwargs)
    
    # Capture post-execution metrics
    end_time = time.perf_counter()
    cpu_times_end = p.cpu_times()
    
    # Get memory info from tracemalloc
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    # Capture final system metrics
    ram_after = psutil.virtual_memory()
    rss_after = p.memory_info().rss / (1024 * 1024)
    
    # Calculate performance metrics
    wall_time = end_time - start_time
    
    # CPU time: sum of user and system time
    cpu_time_used = (cpu_times_end.user + cpu_times_end.system) - \
                    (cpu_times_start.user + cpu_times_start.system)
    
    # CPU percentage: (actual CPU time / wall time) * 100
    # Capped at 100% for display purposes
    cpu_percent = (cpu_time_used / wall_time) * 100 if wall_time > 0 else 0
    
    # Memory metrics
    peak_ram_mb = peak / (1024 * 1024)           # Peak from tracemalloc
    process_rss_mb = max(rss_before, rss_after)  # Maximum RSS
    delta_rss = rss_after - rss_before            # RSS change
    system_ram_percent = ram_after.percent        # System RAM usage %
    
    return {
        'result': result,
        'time': wall_time,
        'cpu_percent': min(cpu_percent, 100.0),
        'cpu_time': cpu_time_used,
        'ram_mb': peak_ram_mb,
        'process_rss_mb': process_rss_mb,
        'delta_rss': delta_rss,
        'system_ram_percent': system_ram_percent
    }

# ============================================================================
# PERFORM CROSS-VALIDATION FOR EACH MODEL
# ============================================================================

print("\nStarting 10-fold cross-validation for all models...")
print("="*80)

for model_name, pipeline in pipelines.items():
    print(f"\n[{model_name}]")
    print("-" * 40)
    
    # Measure cross-validation efficiency and results
    metrics = measure_efficiency(
        cross_validate,
        pipeline, X_train, y_train, 
        cv=skf,              # 10-fold stratified cross-validation
        scoring=scorers,     # All metrics from scorers dict
        n_jobs=-1,           # Use all CPU cores
        return_estimator=False
    )
    
    results = metrics['result']
    
    # Store efficiency metrics
    efficiency_metrics['training_time'].append(metrics['time'])
    efficiency_metrics['cpu_percent'].append(metrics['cpu_percent'])
    efficiency_metrics['ram_mb'].append(metrics['ram_mb'])
    efficiency_metrics['process_rss_mb'].append(metrics['process_rss_mb'])

    # Display computational efficiency
    print(f"Computational Efficiency:")
    print(f"  Wall time:     {metrics['time']:.2f} seconds")
    print(f"  CPU time:      {metrics['cpu_time']:.2f} seconds")
    print(f"  CPU usage:     {metrics['cpu_percent']:.1f}% of wall time")
    print(f"  Peak RAM:      {metrics['ram_mb']:.2f} MB (tracemalloc)")
    print(f"  Process RSS:   {metrics['process_rss_mb']:.2f} MB")
    print(f"  RSS change:    {metrics['delta_rss']:+.2f} MB")
    print(f"  System RAM:    {metrics['system_ram_percent']:.1f}%")
    
    # Store cross-validation results for each fold
    for fold, (acc, f1, prec, rec, roc_auc, mcc) in enumerate(
        zip(results['test_accuracy'], results['test_f1_weighted'],
            results['test_precision'], results['test_recall'], 
            results['test_roc_auc'], results['test_mcc']),
        1
    ):
        cv_results.append({
            'Model': model_name,
            'Fold': fold,
            'Accuracy': acc,
            'F1 Score': f1,
            'Precision': prec,
            'Recall': rec,
            'ROC AUC': roc_auc,
            'MCC': mcc 
        })

# ============================================================================
# SUMMARIZE RESULTS
# ============================================================================

# Create DataFrame with cross-validation results
cv_results_df = pd.DataFrame(cv_results)

# Create efficiency summary DataFrame
efficiency_df = pd.DataFrame({
    'Model': list(pipelines.keys()),
    'Wall Time (s)': efficiency_metrics['training_time'],
    'CPU (%)': efficiency_metrics['cpu_percent'],
    'Peak RAM (MB)': efficiency_metrics['ram_mb'],
    'Process RSS (MB)': efficiency_metrics['process_rss_mb']
})

# Print summaries
print("\n" + "="*80)
print("COMPUTATIONAL EFFICIENCY SUMMARY")
print("="*80)
print(efficiency_df.sort_values('Wall Time (s)').to_string(index=False))

print("\n" + "="*80)
print("MODEL PERFORMANCE SUMMARY (Mean across 10 folds)")
print("="*80)
performance_summary = cv_results_df.groupby('Model')[['Accuracy', 'F1 Score', 'Precision', 'Recall', 'ROC AUC', 'MCC']].mean()
print(performance_summary.sort_values('MCC', ascending=False).to_string())

# ============================================================================
# VISUALIZATION FUNCTIONS
# ============================================================================

def plot_confusion_matrix(y_true, y_pred, title):
    """Plot confusion matrix heatmap"""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    ax = sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                annot_kws={"size": 20},
                xticklabels=["Normal", "Anomaly"],  
                yticklabels=["Normal", "Anomaly"])  
    plt.title(f"Confusion Matrix - {title}", fontsize=20, fontweight='bold')
    plt.xlabel("Predicted Label", fontsize=18, fontweight='bold')
    plt.ylabel("Actual Label", fontsize=18, fontweight='bold')
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=16)
    plt.tight_layout()
    plt.show()

def plot_roc_curve(y_true, y_prob_or_score, title):
    """
    Plot ROC curve for binary classification.
    
    Handles both predict_proba (2D) and decision_function (1D) outputs.
    """
    # Extract probability of positive class
    if len(y_prob_or_score.shape) > 1:  # predict_proba output
        y_score = y_prob_or_score[:, 1]
    else:  # decision_function output
        y_score = y_prob_or_score

    # Calculate ROC curve
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)

    # Plot
    plt.figure(figsize=(10, 8))
    plt.plot(fpr, tpr, color='darkorange', lw=4, 
             label=f'ROC curve (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=4, linestyle='--', 
             label='Random Classifier (AUC = 0.5)')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=18, fontweight='bold')
    plt.ylabel('True Positive Rate', fontsize=18, fontweight='bold')
    plt.title(title, fontsize=20, fontweight='bold')
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)
    plt.legend(loc="lower right", fontsize=16)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# ============================================================================
# PLOT INDIVIDUAL MODEL RESULTS
# ============================================================================

print("\n" + "="*80)
print("GENERATING INDIVIDUAL MODEL VISUALIZATIONS")
print("="*80)

for model_name, pipeline in pipelines.items():
    print(f"\nGenerating plots for: {model_name}")

    # Fit pipeline on training data
    pipeline.fit(X_train, y_train)
    
    # Make predictions on test data
    y_pred = pipeline.predict(X_test)
   
    # Get probability scores for ROC curve
    try:
        # Try predict_proba (for most classifiers)
        y_prob_or_score = pipeline.predict_proba(X_test)
    except AttributeError:
        # Fallback to decision_function (for SVM, LDA, etc.)
        print(f"  Note: Using decision_function for {model_name}")
        y_prob_or_score = pipeline.decision_function(X_test)

    # Plot confusion matrix
    plot_confusion_matrix(y_test, y_pred, f"Confusion Matrix - {model_name}")

    # Plot ROC curve
    plot_roc_curve(y_test, y_prob_or_score, f"ROC Curve - {model_name}")
    
    # Print detailed classification report
    print(f"\nClassification Report for {model_name}:")
    print(classification_report(y_test, y_pred, target_names=['Normal', 'Anomaly']))

# ============================================================================
# COMBINED ROC CURVES COMPARISON
# ============================================================================

print("\n" + "="*80)
print("GENERATING COMBINED ROC CURVES COMPARISON")
print("="*80)

plt.figure(figsize=(12, 8))

# Set the desired order for the models
order = ['MLP', 'Logistic Regression', 'Random Forest', 'KNN',
         'Gradient Boosting', 'Bagging', 'LDA', 'Naive Bayes', 'XGBoost']

# Model name abbreviations for legend
abbr = {
    'Logistic Regression': 'LR',
    'LDA': 'LDA',
    'Bagging': 'Bagging',
    'Naive Bayes': 'NB',
    'KNN': 'KNN',
    'Random Forest': 'RF',
    'Gradient Boosting': 'GBM',
    'XGBoost': 'XGBoost',
    'MLP': 'MLP'
}

# Plot ROC curve for each model
for model_name in order:
    if model_name not in pipelines:
        continue
        
    pipeline = pipelines[model_name]
    pipeline.fit(X_train, y_train)
    
    # Get prediction scores
    try:
        y_prob_or_score = pipeline.predict_proba(X_test)
        y_score = y_prob_or_score[:, 1]
    except AttributeError:
        y_score = pipeline.decision_function(X_test)

    # Calculate and plot ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_auc = auc(fpr, tpr)
    
    label = abbr.get(model_name, model_name)
    plt.plot(fpr, tpr, lw=2.5,
             label=f'{label} (AUC = {roc_auc:.3f})')

# Configure chart
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', alpha=0.5,
         label='Random Classifier (AUC = 0.5)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=14, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=14, fontweight='bold')
plt.title('ROC Curve Comparison - All Models', fontsize=16, fontweight='bold')
plt.legend(loc="lower right", fontsize=11, framealpha=0.95)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ All visualizations completed successfully")


In [ ]:
# ============================================================================
# STEP 10: SAVE THE BEST PERFORMING MODEL
# ============================================================================

print("\n" + "="*80)
print("SAVING THE TRAINED MODEL")
print("="*80)

from joblib import dump

# Save the trained XGBoost pipeline to disk
# The pipeline includes: preprocessor + classifier
model_path = 'XGBoost.joblib'
dump(pipelines['XGBoost'], model_path)



In [ ]:
# ============================================================================
# STEP 11: VERIFY AND TEST THE SAVED MODEL
# ============================================================================

print("\n" + "="*80)
print("VERIFYING SAVED MODEL")
print("="*80)

import joblib
from joblib import load

# Load the saved model
model_path = 'XGBoost.joblib'
loaded_model = joblib.load(model_path)

# Verify pipeline structure
print(f"\n✓ Model loaded successfully from '{model_path}'")
print(f"\nPipeline Components:")
print(f"  Available steps: {list(loaded_model.named_steps.keys())}")

# Display each component
for step_name, step_obj in loaded_model.named_steps.items():
    print(f"  - {step_name}: {type(step_obj).__name__}")

# Verify the classifier type
classifier = loaded_model.named_steps['classifier']
print(f"\nClassifier Details:")
print(f"  Type: {type(classifier).__name__}")
print(f"  Classifier: {classifier}")

